In [19]:
import os
from typing import Optional

from dotenv import load_dotenv
from langchain_core.utils.utils import secret_from_env
from langchain_openai import ChatOpenAI
from pydantic import Field, SecretStr

load_dotenv()

class ChatOpenRouter(ChatOpenAI):
    openai_api_key: Optional[SecretStr] = Field(
        alias="api_key", default_factory=secret_from_env("OPENROUTER_API_KEY", default=None)
    )
    @property
    def lc_secrets(self) -> dict[str, str]:
        return {"openai_api_key": "OPENROUTER_API_KEY"}

    def __init__(self,
                 openai_api_key: Optional[str] = None,
                 **kwargs):
        openai_api_key = openai_api_key or os.environ.get("OPENROUTER_API_KEY")
        super().__init__(base_url="https://openrouter.ai/api/v1", openai_api_key=openai_api_key, **kwargs)


model = ChatOpenRouter(model_name="google/gemini-2.0-flash-001", temperature=0.0)

In [ ]:
# Import relevant functionality
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

# Create the agent
memory = MemorySaver()
tools = []
agent_executor = create_react_agent(model, tools, checkpointer=memory)

In [5]:
import pandas as pd

df = pd.read_csv("/data/chen/data/hmdb/metabolites.csv")

In [13]:
# get 100 rows with 80% endogenous
df_endogenous = df[df['Endogenous']=='Yes'].sample(n=80, random_state=42)
df_non_endogenous = df[df['Endogenous']=='No'].sample(n=20, random_state=42)
df_sampled = pd.concat([df_endogenous, df_non_endogenous]).reset_index(drop=True)


In [14]:
df_sampled

,Accession,Name,Description,Chemical_Formula,Average_Molecular_Weight,Monisotopic_Molecular_Weight,IUPAC_Name,Traditional_IUPAC,CAS_Registry_Number,SMILES,InChI,InChIKey,Synonyms,Ontology,Endogenous
0,HMDB0104373,TG(i-16:0/a-17:0/18:0)[rac],TG(i-16:0/a-17:0/18:0) belongs to the family o...,C54H104O6,849.4200,848.783291,(2R)-2-[(14-methylhexadecanoyl)oxy]-3-[(14-met...,(2R)-2-[(14-methylhexadecanoyl)oxy]-3-[(14-met...,NaN,[H][C@@](COC(=O)CCCCCCCCCCCCCCCCC)(COC(=O)CCCC...,InChI=1S/C54H104O6/c1-6-8-9-10-11-12-13-14-15-...,BVPZRSZXEXQZEN-HUWKAXHXSA-N,1-isohexadecanoyl-2-anteisoheptadecanoyl-3-oct...,\n,Yes
1,HMDB0068318,TG(i-22:0/i-18:0/a-25:0)[rac],TG(i-22:0/i-18:0/a-25:0) belongs to the family...,C68H132O6,1045.7980,1045.002392,(2R)-3-[(20-methylhenicosanoyl)oxy]-2-[(16-met...,(2R)-3-[(20-methylhenicosanoyl)oxy]-2-[(16-met...,NaN,[H][C@@](COC(=O)CCCCCCCCCCCCCCCCCCCCC(C)CC)(CO...,InChI=1S/C68H132O6/c1-7-64(6)56-50-44-38-32-26...,LRRRHCCXZWNSSY-VWDVLKFASA-N,1-isodocosanoyl-2-isooctadecanoyl-3-anteisopen...,\n,Yes
2,HMDB0028681,Alanylarginine,Alanylarginine is a dipeptide composed of alan...,C9H19N5O3,245.2830,245.148789,(2S)-2-[(2S)-2-aminopropanamido]-5-carbamimida...,(2S)-2-[(2S)-2-aminopropanamido]-5-carbamimida...,16709-12-9,C[C@H](N)C(=O)N[C@@H](CCCNC(N)=N)C(O)=O,InChI=1S/C9H19N5O3/c1-5(10)7(15)14-6(8(16)17)3...,SITWEMZOJNKJCH-WDSKDSINSA-N,(2S)-2-{[(2S)-2-amino-1-hydroxypropylidene]ami...,\n,Yes
3,HMDB0058341,"CL(18:1(9Z)/16:0/22:5(4Z,7Z,10Z,13Z,16Z)/18:1(...","CL(18:1(9Z)/16:0/22:5(4Z,7Z,10Z,13Z,16Z)/18:1(...",C83H148O17P2,1480.0280,1479.019177,"[(2R)-3-[(4Z,7Z,10Z,13Z,16Z)-docosa-4,7,10,13,...","(2R)-3-[(4Z,7Z,10Z,13Z,16Z)-docosa-4,7,10,13,1...",NaN,[H][C@](O)(COP(O)(=O)OC[C@@]([H])(COC(=O)CCCCC...,InChI=1S/C83H148O17P2/c1-5-9-13-17-21-25-29-33...,LWOSNUHJAUNLKR-XPBBNWCYSA-N,1'-[1-Oleoyl-2-palmitoyl-sn-glycero-3-phospho]...,\n,Yes
4,HMDB0096061,TG(12:0/i-20:0/14:0),TG(12:0/i-20:0/14:0) belongs to the family of ...,C49H94O6,779.2850,778.705041,(2S)-1-(dodecanoyloxy)-3-(tetradecanoyloxy)pro...,(2S)-1-(dodecanoyloxy)-3-(tetradecanoyloxy)pro...,NaN,[H][C@](COC(=O)CCCCCCCCCCC)(COC(=O)CCCCCCCCCCC...,InChI=1S/C49H94O6/c1-5-7-9-11-13-15-20-25-29-3...,ZQCNILQMSQAMBS-DXQCBLCSSA-N,1-dodecanoyl-2-isoeicosanoyl-3-tetradecanoyl-g...,\n,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,HMDB0015621,Sulfadimethoxine,"Sulfadimethoxine (trade name Di-Methox, Albon)...",C12H14N4O4S,310.3290,310.073576,"4-amino-N-(2,6-dimethoxypyrimidin-4-yl)benzene...",sulfadimethoxine,122-11-2,COC1=NC(OC)=NC(NS(=O)(=O)C2=CC=C(N)C=C2)=C1,InChI=1S/C12H14N4O4S/c1-19-11-7-10(14-12(15-11...,ZZORFUFYDOWNEF-UHFFFAOYSA-N,"2,4-Dimethoxy-6-sulfanilamido-1,3-diazine; 2,6...",\n,No
96,HMDB0040597,Eduline,Eduline is found in pomes. Eduline is an alkal...,C17H15NO2,265.3065,265.110279,"6-methoxy-1-methyl-2-phenyl-1,4-dihydroquinoli...",6-methoxy-1-methyl-2-phenylquinolin-4-one,6878-08-6,COC1=CC2=C(C=C1)N(C)C(=CC2=O)C1=CC=CC=C1,InChI=1S/C17H15NO2/c1-18-15-9-8-13(20-2)10-14(...,GFUAPSNFZWUMBP-UHFFFAOYSA-N,"6-Methoxy-1-methyl-2-phenyl-4(1H)-quinolinone,...",\n,No
97,HMDB0039373,6''-(4-Carboxy-3-hydroxy-3-methylbutanoyl)hyperin,6''-(4-Carboxy-3-hydroxy-3-methylbutanoyl)hype...,C27H28O16,608.5016,608.137735,"5-[(6-{[2-(3,4-dihydroxyphenyl)-5,7-dihydroxy-...","5-[(6-{[2-(3,4-dihydroxyphenyl)-5,7-dihydroxy-...",106915-93-9,CC(O)(CC(O)=O)CC(=O)OCC1OC(OC2=C(OC3=CC(O)=CC(...,"InChI=1S/C27H28O16/c1-27(39,7-17(32)33)8-18(34...",FMMBDZGNMFRGMV-UHFFFAOYSA-N,Quercetin 3-[6''-(3-hydroxy-3-methylglutaryl)g...,\n,No
98,HMDB0033537,Fragransol A,Fragransol A is found in herbs and spices. Fra...,C21H26O6,374.4275,374.172939,4-[5-(2-hydroxy-1-methoxypropyl)-7-methoxy-3-m...,4-[5-(2-hydroxy-1-methoxypropyl)-7-methoxy-3-m...,114394-19-3,COC(C(C)O)C1=CC2=C(OC(C2C)C2=CC(OC)=C(O)C=C2)C...,InChI=1S/C21H26O6/c1-11-15-8-14(20(26-5)12(2)2...,DHRVKFSLKMAPEL-UHFFFAOYSA-N,Digestion; Cellular membrane; Flora,\n,No


In [21]:
# use model to take input compound name and return if it is endogenous metabolite from human
def is_endogenous_compound(compound_name: str) -> str:
    prompt = f"Is the compound '{compound_name}' an endogenous metabolite in humans? Answer 'Yes', 'No'."
    response = model.predict(prompt)
    return response.strip()

df_sampled['Endogenous_LLM'] = df_sampled['Name'].apply(is_endogenous_compound)
df_sampled[['Name', 'Endogenous', 'Endogenous_LLM']]

,Name,Endogenous,Endogenous_LLM
0,TG(i-16:0/a-17:0/18:0)[rac],Yes,No
1,TG(i-22:0/i-18:0/a-25:0)[rac],Yes,No
2,Alanylarginine,Yes,Yes
3,"CL(18:1(9Z)/16:0/22:5(4Z,7Z,10Z,13Z,16Z)/18:1(...",Yes,Yes
4,TG(12:0/i-20:0/14:0),Yes,No
...,...,...,...
95,Sulfadimethoxine,No,No
96,Eduline,No,No
97,6''-(4-Carboxy-3-hydroxy-3-methylbutanoyl)hyperin,No,No
98,Fragransol A,No,No


In [24]:
df_sampled.to_csv("data/metabolites_llm_labled.csv", index=False)

In [2]:
import pandas as pd

df_sampled = pd.read_csv("data/metabolites_llm_labled.csv")

In [4]:
df_sampled

,Accession,Name,Description,Chemical_Formula,Average_Molecular_Weight,Monisotopic_Molecular_Weight,IUPAC_Name,Traditional_IUPAC,CAS_Registry_Number,SMILES,InChI,InChIKey,Synonyms,Ontology,Endogenous,Endogenous_LLM
0,HMDB0104373,TG(i-16:0/a-17:0/18:0)[rac],TG(i-16:0/a-17:0/18:0) belongs to the family o...,C54H104O6,849.4200,848.783291,(2R)-2-[(14-methylhexadecanoyl)oxy]-3-[(14-met...,(2R)-2-[(14-methylhexadecanoyl)oxy]-3-[(14-met...,NaN,[H][C@@](COC(=O)CCCCCCCCCCCCCCCCC)(COC(=O)CCCC...,InChI=1S/C54H104O6/c1-6-8-9-10-11-12-13-14-15-...,BVPZRSZXEXQZEN-HUWKAXHXSA-N,1-isohexadecanoyl-2-anteisoheptadecanoyl-3-oct...,\n,Yes,No
1,HMDB0068318,TG(i-22:0/i-18:0/a-25:0)[rac],TG(i-22:0/i-18:0/a-25:0) belongs to the family...,C68H132O6,1045.7980,1045.002392,(2R)-3-[(20-methylhenicosanoyl)oxy]-2-[(16-met...,(2R)-3-[(20-methylhenicosanoyl)oxy]-2-[(16-met...,NaN,[H][C@@](COC(=O)CCCCCCCCCCCCCCCCCCCCC(C)CC)(CO...,InChI=1S/C68H132O6/c1-7-64(6)56-50-44-38-32-26...,LRRRHCCXZWNSSY-VWDVLKFASA-N,1-isodocosanoyl-2-isooctadecanoyl-3-anteisopen...,\n,Yes,No
2,HMDB0028681,Alanylarginine,Alanylarginine is a dipeptide composed of alan...,C9H19N5O3,245.2830,245.148789,(2S)-2-[(2S)-2-aminopropanamido]-5-carbamimida...,(2S)-2-[(2S)-2-aminopropanamido]-5-carbamimida...,16709-12-9,C[C@H](N)C(=O)N[C@@H](CCCNC(N)=N)C(O)=O,InChI=1S/C9H19N5O3/c1-5(10)7(15)14-6(8(16)17)3...,SITWEMZOJNKJCH-WDSKDSINSA-N,(2S)-2-{[(2S)-2-amino-1-hydroxypropylidene]ami...,\n,Yes,Yes
3,HMDB0058341,"CL(18:1(9Z)/16:0/22:5(4Z,7Z,10Z,13Z,16Z)/18:1(...","CL(18:1(9Z)/16:0/22:5(4Z,7Z,10Z,13Z,16Z)/18:1(...",C83H148O17P2,1480.0280,1479.019177,"[(2R)-3-[(4Z,7Z,10Z,13Z,16Z)-docosa-4,7,10,13,...","(2R)-3-[(4Z,7Z,10Z,13Z,16Z)-docosa-4,7,10,13,1...",NaN,[H][C@](O)(COP(O)(=O)OC[C@@]([H])(COC(=O)CCCCC...,InChI=1S/C83H148O17P2/c1-5-9-13-17-21-25-29-33...,LWOSNUHJAUNLKR-XPBBNWCYSA-N,1'-[1-Oleoyl-2-palmitoyl-sn-glycero-3-phospho]...,\n,Yes,Yes
4,HMDB0096061,TG(12:0/i-20:0/14:0),TG(12:0/i-20:0/14:0) belongs to the family of ...,C49H94O6,779.2850,778.705041,(2S)-1-(dodecanoyloxy)-3-(tetradecanoyloxy)pro...,(2S)-1-(dodecanoyloxy)-3-(tetradecanoyloxy)pro...,NaN,[H][C@](COC(=O)CCCCCCCCCCC)(COC(=O)CCCCCCCCCCC...,InChI=1S/C49H94O6/c1-5-7-9-11-13-15-20-25-29-3...,ZQCNILQMSQAMBS-DXQCBLCSSA-N,1-dodecanoyl-2-isoeicosanoyl-3-tetradecanoyl-g...,\n,Yes,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,HMDB0015621,Sulfadimethoxine,"Sulfadimethoxine (trade name Di-Methox, Albon)...",C12H14N4O4S,310.3290,310.073576,"4-amino-N-(2,6-dimethoxypyrimidin-4-yl)benzene...",sulfadimethoxine,122-11-2,COC1=NC(OC)=NC(NS(=O)(=O)C2=CC=C(N)C=C2)=C1,InChI=1S/C12H14N4O4S/c1-19-11-7-10(14-12(15-11...,ZZORFUFYDOWNEF-UHFFFAOYSA-N,"2,4-Dimethoxy-6-sulfanilamido-1,3-diazine; 2,6...",\n,No,No
96,HMDB0040597,Eduline,Eduline is found in pomes. Eduline is an alkal...,C17H15NO2,265.3065,265.110279,"6-methoxy-1-methyl-2-phenyl-1,4-dihydroquinoli...",6-methoxy-1-methyl-2-phenylquinolin-4-one,6878-08-6,COC1=CC2=C(C=C1)N(C)C(=CC2=O)C1=CC=CC=C1,InChI=1S/C17H15NO2/c1-18-15-9-8-13(20-2)10-14(...,GFUAPSNFZWUMBP-UHFFFAOYSA-N,"6-Methoxy-1-methyl-2-phenyl-4(1H)-quinolinone,...",\n,No,No
97,HMDB0039373,6''-(4-Carboxy-3-hydroxy-3-methylbutanoyl)hyperin,6''-(4-Carboxy-3-hydroxy-3-methylbutanoyl)hype...,C27H28O16,608.5016,608.137735,"5-[(6-{[2-(3,4-dihydroxyphenyl)-5,7-dihydroxy-...","5-[(6-{[2-(3,4-dihydroxyphenyl)-5,7-dihydroxy-...",106915-93-9,CC(O)(CC(O)=O)CC(=O)OCC1OC(OC2=C(OC3=CC(O)=CC(...,"InChI=1S/C27H28O16/c1-27(39,7-17(32)33)8-18(34...",FMMBDZGNMFRGMV-UHFFFAOYSA-N,Quercetin 3-[6''-(3-hydroxy-3-methylglutaryl)g...,\n,No,No
98,HMDB0033537,Fragransol A,Fragransol A is found in herbs and spices. Fra...,C21H26O6,374.4275,374.172939,4-[5-(2-hydroxy-1-methoxypropyl)-7-methoxy-3-m...,4-[5-(2-hydroxy-1-methoxypropyl)-7-methoxy-3-m...,114394-19-3,COC(C(C)O)C1=CC2=C(OC(C2C)C2=CC(OC)=C(O)C=C2)C...,InChI=1S/C21H26O6/c1-11-15-8-14(20(26-5)12(2)2...,DHRVKFSLKMAPEL-UHFFFAOYSA-N,Digestion; Cellular membrane; Flora,\n,No,No


In [6]:
# calculate accuracy, precision, recall and F1 score
df_sampled['Endogenous_bool'] = df_sampled['Endogenous'] == 'Yes'
df_sampled['Endogenous_LLM_bool'] = df_sampled['Endogenous_LLM'] == 'Yes'

accuracy = (df_sampled['Endogenous_bool'] == df_sampled['Endogenous_LLM_bool']).mean()
precision = (df_sampled['Endogenous_bool'] & df_sampled['Endogenous_LLM_bool']).sum() / df_sampled['Endogenous_LLM_bool'].sum()
recall = (df_sampled['Endogenous_bool'] & df_sampled['Endogenous_LLM_bool']).sum() / df_sampled['Endogenous_bool'].sum()
f1 = 2 * (precision * recall) / (precision + recall)

accuracy, precision, recall, f1

(0.34, 0.8888888888888888, 0.2, 0.326530612244898)